> **Chapter 10, Part 0** | Bridge notebook. **Focus:** retrieval evaluation, hybrid search, and reranking before the answer layer ever speaks.


# Retrieval Evaluation, Hybrid Search, and Reranking

A retrieval system usually fails long before answer synthesis. The failure mode is not always that the model is weak. More often, the wrong chunk won.

This notebook exists because learners need one clean stop between vector search and grounded answers. It is the place where ranking becomes inspectable rather than mystical.

## Outputs

- a toy retrieval set with semantic and lexical signals
- a simple `precision_at_k` calculation
- a hybrid ranking that combines vector similarity and keyword overlap
- a reranking pass that boosts citation-ready candidates with useful metadata

## Reading pack

- precision at k, recall at k, and why one metric never settles retrieval quality
- hybrid search as a practical compromise rather than a theoretical luxury
- reranking as a narrow ranking correction, not a second full answer model

## Failure note

If you do not know what a bad top-3 result looks like, you cannot tell whether the answer layer is broken or the retriever never had a chance.

## How I would debug this

I print the semantic ranking first, then the hybrid ranking, then the reranked list. If the right chunk only appears after reranking, the retriever is still weak even if the final answer looks good.


In [ ]:
import math
import pandas as pd

rows = pd.DataFrame(
    [
        {
            "chunk_id": "c1",
            "title": "Accessible riverside stop",
            "snippet": "A paved route with shuttle access makes this a strong first stop.",
            "semantic_score": 0.92,
            "keyword_overlap": 2,
            "metadata_bonus": 0.04,
            "relevant": True,
        },
        {
            "chunk_id": "c2",
            "title": "General shuttle operations",
            "snippet": "Shuttle timing and stop intervals for mid-day visitors.",
            "semantic_score": 0.88,
            "keyword_overlap": 1,
            "metadata_bonus": 0.00,
            "relevant": False,
        },
        {
            "chunk_id": "c3",
            "title": "Trail safety briefing",
            "snippet": "Steep sections, weather changes, and uneven ground warnings.",
            "semantic_score": 0.81,
            "keyword_overlap": 0,
            "metadata_bonus": 0.00,
            "relevant": False,
        },
        {
            "chunk_id": "c4",
            "title": "Accessible visitor guide",
            "snippet": "Low-grade walking route, paved access, and signage for families.",
            "semantic_score": 0.84,
            "keyword_overlap": 3,
            "metadata_bonus": 0.06,
            "relevant": True,
        },
        {
            "chunk_id": "c5",
            "title": "Evening ranger program",
            "snippet": "Talk schedule, amphitheater location, and evening topics.",
            "semantic_score": 0.73,
            "keyword_overlap": 0,
            "metadata_bonus": 0.00,
            "relevant": False,
        },
    ]
)

rows


In [ ]:
def precision_at_k(frame, k):
    top = frame.head(k)
    return top["relevant"].mean()


def min_max_scale(series):
    spread = series.max() - series.min()
    if spread == 0:
        return series * 0
    return (series - series.min()) / spread


semantic_ranked = rows.sort_values("semantic_score", ascending=False).reset_index(drop=True)
semantic_ranked[["chunk_id", "title", "semantic_score", "relevant"]]


In [ ]:
hybrid_ranked = rows.assign(
    keyword_scaled=min_max_scale(rows["keyword_overlap"]),
).assign(
    hybrid_score=lambda frame: 0.7 * frame["semantic_score"] + 0.3 * frame["keyword_scaled"]
).sort_values("hybrid_score", ascending=False).reset_index(drop=True)

hybrid_ranked[["chunk_id", "title", "semantic_score", "keyword_overlap", "hybrid_score", "relevant"]]


In [ ]:
reranked = hybrid_ranked.assign(
    rerank_score=lambda frame: frame["hybrid_score"] + frame["metadata_bonus"]
).sort_values("rerank_score", ascending=False).reset_index(drop=True)

comparison = pd.DataFrame(
    [
        {"stage": "semantic", "precision_at_1": precision_at_k(semantic_ranked, 1), "precision_at_3": precision_at_k(semantic_ranked, 3)},
        {"stage": "hybrid", "precision_at_1": precision_at_k(hybrid_ranked, 1), "precision_at_3": precision_at_k(hybrid_ranked, 3)},
        {"stage": "reranked", "precision_at_1": precision_at_k(reranked, 1), "precision_at_3": precision_at_k(reranked, 3)},
    ]
)

comparison


In [ ]:
reranked[["chunk_id", "title", "hybrid_score", "metadata_bonus", "rerank_score", "relevant"]]


## Reading the result

The semantic ranking is not wrong, but it is incomplete. It likes the general shuttle chunk because the vector is close enough. The hybrid rank improves things by giving lexical overlap a modest vote. The reranker improves them again because the accessible-guide chunks also carry the metadata we care about for citation-ready answers.

That is the practical lesson. Reranking should be a narrow intervention that helps the right chunk win more often. If it is doing all the real work, the first-stage retriever is not ready.

## Why this belongs before 10.1 and 10.5

- before `10.1`, because learners need a ranking vocabulary before they talk about evidence flow
- before `10.5`, because grounded answers depend on a ranking process that deserves inspection in its own right

## Exercise

1. change the hybrid weighting from `0.7 / 0.3` to something else and inspect the effect
2. change the metadata bonus so it helps one irrelevant chunk and ask whether the reranker became too aggressive
3. replace `precision_at_k` with recall at k and compare what each metric hides
